# 3 Loss, optimizer, and the loop

Notebook 01 gave you vectors, notebook 02 gave you the layer. This one closes the gap
between them: how a cache becomes batches, how you tell the model it is wrong, how it
changes its mind, and how you tell when it has learned enough.

None of this is new in 2026 — it is the machinery every course teaches. What is different
is that you already have something worth training on, so we can go through it once,
properly, at speed, and end with a **number that the rest of the lesson has to beat.**

That last part matters more than it sounds. Notebook 04 runs a search over dozens of
architectures. A search without a baseline established first is unreadable — you cannot
tell a win from a wobble.

In [ ]:
import sys
import time
from pathlib import Path

import datasets
import torch
from loguru import logger
from torch import nn
from torch.utils.data import DataLoader
from vectormesh import VectorCache

from notebooktester import param

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()

torch.manual_seed(42)

train_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="train")
test_cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")

HIDDEN = train_cache.dim  # read the metadata, don't guess
N_CLASSES = train_cache.num_classes

print(f"{len(train_cache)} train / {len(test_cache)} test rows")
print(f"hidden_size {HIDDEN}, {N_CLASSES} classes")

## 3.1 From cache rows to batches

A cache row is a dict: one vector, a 1D tensor `(D)`, and one label, which is just the
number of the class, e.g. 5. A model wants a **batch**: one `(B, D)` 2D tensor of inputs,
e.g. with `B=32` there are 32 of those 1D tensors stacked on top of each other, and one
tensor of `B` targets. This is the `(D) → (B, D)` step
[01b_tensors.ipynb](01b_tensors.ipynb) §2 drew with five movie reviews: separate vectors,
stacked into one tensor. The function that does this is called a *collate function*, and
it is worth writing once by hand so it stops being magic.

In [ ]:
rows = [train_cache[i] for i in range(4)]
print("one row:", {k: tuple(v.shape) for k, v in rows[0].items()})


def collate_by_hand(rows: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    X = torch.stack([row["embed"] for row in rows])  # list of (D) -> (B, D)
    y = torch.stack([row["label"] for row in rows])  # list of ()  -> (B,)
    return X, y


X, y = collate_by_hand(rows)
print(f"\n{len(rows)} rows -> X {tuple(X.shape)}, y {tuple(y.shape)}")

In [ ]:
import matplotlib.pyplot as plt

from plotting import plot_stacking

class_names = train_cache.dataset.features["label"].names
labels = [
    f"cache row {i}: a tile of class {int(row['label'])}, '{class_names[int(row['label'])]}'"
    for i, row in enumerate(rows)
]
plot_stacking(
    [row["embed"] for row in rows],
    labels,
    kind="batch",
    title="Batching is stacking: four cache rows become one (B, D) tensor",
)
plt.show()

The same drawing as the batch of reviews in notebook 01b, now with EuroSAT tiles: four
independent items, four separate `(D)` vectors, one `torch.stack`. The labels go through
the same step on their own: four `()` scalars become one `(B,)` tensor.

That is the whole job: **a list of items becomes a pair of tensors.** Everything else a
collate function ever does is a variation on it — padding (adding zeros when you need to batch data with different sizes, eg sentences that are sometimes shorter, sometimes longer) when the items have different
lengths, stacking when they do not.

Ours is the easy case. `tensordtype: 1` in the metadata told us each item is `(D)`, so
`torch.stack` is enough. Lesson 5's chunked documents are `(C, D)` where every document is chunked into pieces of 512 tokens, giving a different `C` because the documents have different lengths, and *there* the padder has to do real work.

One more conversion happens on the way: the label. `train_cache["label"]` is a single
integer per row — class 5, say — but a classifier's job is not to name one class, it is to
output a **probability distribution over all of them**: "30% forest, 50% river, 20%
everything else." Training needs a target in that same shape to compare against, and
one-hot encoding is the simplest possible one: 100% on the true class, 0% everywhere else.
It is not the only choice — label smoothing spreads a little probability onto the other
classes on purpose, and lesson 5's documents can be genuinely multi-label — but single-class
one-hot is the right target for a task like this one, where exactly one class is correct.

So use the library version, which is the same idea with the shapes checked:

### Three splits: train, valid, test

The cache comes with two splits, train and test. Training needs a third one. While you
train you will make decisions by looking at a split's score: when to stop, which model to
keep. Every decision like that fits your choices a little to that split, so its score ends
up a little too good. If you made them on test, the test score would no longer say how the
model does on data it has never been tuned for, and that is the only thing a test score is
for.

So the roles are:

| split | the model trains on it | you decide on it | you look at it |
|---|---|---|---|
| **train** | yes | no | every epoch |
| **valid** | no | yes: when to stop, which model wins | every epoch |
| **test** | no | no | once, at the end |

EuroSAT's cache has no valid split, so we carve one out of train: 20% of its rows, drawn
per class (`stratify_by_column="label"`) so every class keeps its share, with a fixed seed
so you get the same rows every time you run the notebook. Slide 43 shows the train and
valid curves this split makes possible.

In [ ]:
from vectormesh.data import Collate, OneHot

# CrossEntropyLoss accepts soft targets, so we one-hot the label into its own column
onehot = OneHot(num_classes=N_CLASSES, label_col="label", target_col="onehot")
collate_fn = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)

# 80% of the train rows to train on, 20% to decide on; test stays untouched
split = train_cache.dataset.train_test_split(
    test_size=0.2, seed=42, stratify_by_column="label"
)
trainset, validset = split["train"], split["test"]
testset = test_cache.dataset

BATCH = 64
trainloader = DataLoader(
    trainset.map(onehot), batch_size=BATCH, shuffle=True, collate_fn=collate_fn
)
validloader = DataLoader(
    validset.map(onehot), batch_size=BATCH, shuffle=False, collate_fn=collate_fn
)
testloader = DataLoader(
    testset.map(onehot), batch_size=BATCH, shuffle=False, collate_fn=collate_fn
)

X, y = next(iter(trainloader))
print(f"X {tuple(X.shape)}  {X.dtype}")
print(f"y {tuple(y.shape)}  {y.dtype}   <- one-hot, so (B, K) not (B,)")
print(f"{len(trainset)} train / {len(validset)} valid / {len(testset)} test rows")
print(
    f"{len(trainloader)} train batches, {len(validloader)} valid, {len(testloader)} test"
)

`(B, D) → (B, K)`. That is the entire trip from notebook 01's ladder, and you are now
standing on the second rung.

Note the loaders differ in one flag. `shuffle=True` on train, because gradient descent
on a fixed order learns the order; `shuffle=False` on valid and test, because nothing is
learned there and you want a reproducible number.

## 3.2 Losses: what "wrong" means

A loss is a single number saying how badly the model did, differentiable so that we can ask
which direction reduces it. Picking one is not a detail — it *defines* the task.

| task | one item's target | loss | metric |
|---|---|---|---|
| regression | a real number | `MSELoss` / `L1Loss` | MAE, MASE |
| **single-label** classification | exactly one of K | **`CrossEntropyLoss`** | **Accuracy** |
| multi-label classification | any subset of K | `BCEWithLogitsLoss` | F1 |

**Our task is the middle row.** A EuroSAT tile is a forest or a river, not both, so the ten
scores should compete with each other — which is exactly what the softmax inside
`CrossEntropyLoss` does. Accuracy is the honest metric because the classes are roughly
balanced.

The distinction to hold on to is *single-label vs multi-label*, and it is about the data,
not the model:

### Loss and metric are not the same thing, even when they look alike

A **loss** exists for the optimizer: it has to be differentiable everywhere it matters,
because `backward()` needs a gradient to follow. A **metric** exists for you: it has to be
interpretable, and it does not need a gradient at all — `Accuracy` is a step function
(right or wrong, nothing in between), which is exactly why it cannot be the loss.
`CrossEntropyLoss` is the smooth stand-in that *does* have a gradient and points in roughly
the direction accuracy would improve.

They share one requirement, though: **both need to define what "good" means for this task**,
and a metric can lie about that as easily as a loss can. A few you will use in this course,
beyond the classification pair above:

- **MAE** (mean absolute error) — a regression metric in the same units as the target, so
  "MAE of 2.3 degrees" means something a raw MSE (squared, unitless) does not. Also a valid
  loss (`L1Loss`) if you want the optimizer to weight all errors equally rather than
  squaring the large ones.
- **MASE** (mean absolute scaled error) — MAE divided by the MAE of the dumbest reasonable
  baseline (predict "same as last time"). `MASE < 1` means you beat that baseline;
  `MASE > 1` means a naive guess would have done better. Lesson 3's time-series models are
  judged this way, because a raw MAE number means nothing without knowing whether the
  series was easy or hard to predict in the first place.
- **F1** — the harmonic mean of precision and recall, used instead of accuracy when classes
  are imbalanced or when both false positives and false negatives cost something.
  Lesson 5's multi-label documents use it: a document can be tagged with several labels, so
  "percent exactly right" is far too strict a metric, and F1 rewards getting *most* of the
  labels right per document.

The pattern under all of them: a loss is chosen so the optimizer can follow it; a metric is
chosen so a human can trust it. Sometimes one formula does both jobs (MAE); often it takes
two (CrossEntropy + Accuracy, BCEWithLogits + F1).

In [ ]:
logits = torch.tensor([2.0, 1.0, 0.1])


def show(name, scores, comment):
    print(
        f"{name:24s} [{', '.join(f'{s:.3f}' for s in scores)}]   sums to {scores.sum():.3f}   {comment}"
    )


show("raw logits", logits, "")
show("softmax (CrossEntropy)", torch.softmax(logits, dim=0), "<- the classes compete")
show("sigmoid (BCEWithLogits)", torch.sigmoid(logits), "<- each class decided alone")

Raising one softmax score pushes the others down; raising one sigmoid score does nothing to
its neighbours. That is the whole difference, and it is why lesson 5's legal-document task
takes `BCEWithLogits` + F1: a ruling can be about tenancy law *and* liability at once, so
forcing those two labels to compete would be a modelling error, not a tuning choice.

Two practical notes that cost people hours:

- **`CrossEntropyLoss` eats logits, not probabilities.** Do not put a softmax at the end of
  your model — the loss applies its own, and doing it twice quietly flattens your gradients.
- **`BCEWithLogits` is `Sigmoid + BCELoss` fused**, and the fused version is the numerically
  stable one. Use it rather than assembling the pieces.

## 3.3 The optimizer, and the loop

Backpropagation gives you $\partial \text{loss} / \partial w$ for every weight: which way to
nudge each one. The optimizer decides how far to step. `Adam` adapts the step size per
parameter and is a fine default for everything in this course.

### What an optimizer does, drawn

A real model has thousands of weights, so its loss is a surface you cannot draw. Shrink the
model to two weights and you can. The surface below is a valley: steep along weight 2,
nearly flat along weight 1. Real networks have this shape too (some weights move the loss
a lot, others barely), and it is where optimizers differ.

Every run starts at the star and takes 40 steps toward the minimum at the cross. The loop
inside `descend` is the same `zero_grad → loss → backward → step` loop you will write for
the real model below; only the model is smaller.

In [ ]:
from plotting import plot_optimizer_paths


def valley(w: torch.Tensor) -> torch.Tensor:
    """A loss over two weights: steep along weight 2, nearly flat along weight 1."""
    return 0.1 * w[0] ** 2 + 2 * w[1] ** 2


def descend(make_optimizer, steps: int = 40) -> torch.Tensor:
    w = torch.tensor(
        [-4.0, 2.0], requires_grad=True
    )  # the two weights, at the starting point
    optimizer = make_optimizer([w])
    path = [w.detach().clone()]
    for _ in range(steps):
        optimizer.zero_grad()
        loss = valley(w)
        loss.backward()
        optimizer.step()
        path.append(w.detach().clone())
    return torch.stack(path)  # (steps + 1, 2): one row per step


fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_optimizer_paths(
    valley,
    {
        "SGD, lr=0.2": descend(lambda w: torch.optim.SGD(w, lr=0.2)),
        "SGD + momentum, lr=0.2": descend(
            lambda w: torch.optim.SGD(w, lr=0.2, momentum=0.7)
        ),
        "Adam, lr=0.3": descend(lambda w: torch.optim.Adam(w, lr=0.3)),
    },
    ax=axes[0],
    title="One valley, three optimizers",
)
plot_optimizer_paths(
    valley,
    {
        "SGD, lr=0.02": descend(lambda w: torch.optim.SGD(w, lr=0.02)),
        "SGD, lr=0.3": descend(lambda w: torch.optim.SGD(w, lr=0.3)),
        "SGD, lr=0.505": descend(lambda w: torch.optim.SGD(w, lr=0.505)),
    },
    ax=axes[1],
    title="SGD, three learning rates",
)
plt.tight_layout()
plt.show()

**Left, the optimizers.** Plain **SGD** steps along the gradient, times the learning
rate. It drops into the valley fast, because the steep direction has a large gradient,
and then crawls along the flat floor, where the gradient is tiny. **Momentum** adds a
running sum of earlier steps, so it picks up speed along the floor; the price is
overshooting, the swings across the valley. **Adam** divides each weight's step by the
recent size of that weight's own gradient, so both weights move at a similar pace whatever
the slope: that is why its first steps go diagonally. One learning rate then works for
weights whose gradients differ a lot in size, which is what makes Adam a good default. It
does not make Adam the winner everywhere: on this valley momentum gets closest to the
minimum (see the losses in the legend), and on another surface the order changes.

**Right, the learning rate.** The learning rate is the step size, and it has to fit the
surface. Too small (0.02): after 40 steps SGD has hardly moved. Too large (0.505): each
step overshoots the valley floor by more than the last, so the path bounces higher and the
loss grows instead of shrinking. In between (0.3) it settles into the valley and moves toward the minimum. On a real network you cannot
see the surface, only the loss curve: a loss that barely falls, or one that jumps around
and ends in `nan`, are these two failures.

### The loop

Here is the loop for the real model, by hand, once. Five lines, in this order, and getting
the order wrong is one of the classic silent bugs. Three epochs are enough to watch the
numbers move; how many epochs a model needs is the question of §3.4. After every epoch
the loop scores the valid split, the one you decide on.

```mermaid
flowchart LR
    A["1 zero_grad()<br/>clear old gradients"] --> B["2 forward: model(X)<br/>-> yhat"]
    B --> C["3 loss_fn(yhat, y)<br/>-> one number"]
    C --> D["4 loss.backward()<br/>gradient for every weight"]
    D --> E["5 optimizer.step()<br/>nudge the weights"]
    E -.next batch.-> A
```

Forward computes a prediction and a loss; backward turns that one number into a gradient
for every weight in the model, by the chain rule; `step()` uses those gradients to move the
weights. Nothing here is specific to this task — swap the model, the loss, or the data and
the five-line shape stays exactly the same.

In [ ]:
model = nn.Linear(HIDDEN, N_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(param(3, test=1)):
    total = 0.0
    for X, y in trainloader:
        optimizer.zero_grad()  # 1. clear last batch's gradients -- they ACCUMULATE
        yhat = model(X)  # 2. prediction (forward)
        loss = loss_fn(yhat, y)  # 3. how wrong was the prediction?
        loss.backward()  # 4. gradient of the loss w.r.t. every weight (backward)
        optimizer.step()  # 5. nudge the weights
        total += float(loss.detach())

    model.eval()
    with torch.no_grad():
        correct = sum(
            float((model(X).argmax(1) == y.argmax(1)).sum()) for X, y in validloader
        )
    model.train()
    print(
        f"epoch {epoch}  train loss {total / len(trainloader):.4f}  valid acc {correct / len(validset):.4f}"
    )

Things worth naming in that loop, because you will meet all of them as bugs:

- **`zero_grad()` first.** Gradients accumulate by default. Forget it and every step uses
  the sum of all gradients so far.
- **`no_grad()` for evaluation.** No gradients are needed, and it halves the memory.
- **`.eval()` / `.train()`.** Some layers behave differently in the two modes — dropout,
  batchnorm, and `GaussianNoise`, which you will meet in notebook 04. Evaluating in train
  mode gives you a noisy, irreproducible number.

That is the last time you will write this loop. It is short enough to understand and
repetitive enough that retyping it per experiment is how mistakes get in. From here on,
`vectormesh.training.Trainer` runs it, with the same five steps and the same
`.eval()`/`.train()`/`no_grad()` discipline handled for you.

The `train` function below wraps it. Three things it adds to the loop above:

- after every epoch it computes the loss on the valid split, the *valid loss*, and a small
  `record` function (a reporter: `Trainer` calls it once per epoch) keeps the train loss,
  the valid loss and the learning rate, so you can draw the curves;
- `epochs` is the largest number of epochs it may run, `EPOCHS`;
- `patience` switches early stopping on or off; §3.5 explains it, until then it is off.

`score` computes accuracy on any loader. The test loader goes through it once, in §3.6.

In [ ]:
from vectormesh.components import NeuralNet, Serial
from vectormesh.components.metrics import Accuracy
from vectormesh.data.vectorizers import detect_device
from vectormesh.training import Trainer, TrainerSettings

EPOCHS = param(30, test=2)  # the most epochs a run may take
device = detect_device()


def train(
    model, patience: int | None = None, epochs: int = EPOCHS, trainloader=trainloader
):
    """Train on trainloader, evaluate on validloader after every epoch.

    patience=None runs every epoch; an int switches on early stopping.
    Returns the Trainer's result, the curves per epoch and the seconds it took.
    """
    early = (
        None
        if patience is None
        else {"patience": patience, "save": False, "verbose": False}
    )
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[Accuracy()],
        logdir=Path("modellogs").resolve(),
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        earlystop_kwargs=early,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.CrossEntropyLoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
        progress=False,
    )
    curves = {"train_loss": [], "valid_loss": [], "lr": []}

    def record(epoch, train_loss, valid_loss, metric_dict):
        curves["train_loss"].append(train_loss)
        curves["valid_loss"].append(valid_loss)
        curves["lr"].append(trainer.optimizer.param_groups[0]["lr"])

    trainer.reporters.append(record)
    start = time.time()
    result = trainer.loop()  # with early stopping: the best epoch, its weights restored
    return result, curves, time.time() - start


def score(model, loader) -> float:
    """Accuracy of model on every row of loader."""
    model.eval()
    with torch.no_grad():
        right = sum(
            float((model(X.to(device)).argmax(1) == y.to(device).argmax(1)).sum())
            for X, y in loader
        )
    return right / len(loader.dataset)


print(f"training on {device}")

## 3.4 How long to train: read the curve

How many epochs does a model need? Any number you type is a guess: too few and the model
had not finished, too many and it starts to memorise the training rows. You do not have to
guess, because the model tells you. Train for longer than it needs, and look at two curves
per epoch: the **train loss** (on the rows it learns from) and the **valid loss** (on rows
it never learns from).

Three shapes to recognise, the same ones slide 43 shows:

- **still learning**: both curves are still falling at the last epoch; more epochs would help;
- **finished**: both curves are flat; more epochs only cost time;
- **overfitting**: the train loss keeps falling while the valid loss turns up; the model is
  getting better at the training rows and worse at new ones.

The model is the one of §3.6, `NeuralNet`: two `Linear` layers with a `GELU` between
them, notebook 02's construction. It trains for all `EPOCHS` = 30 epochs, early stopping off.

**Before you run it, write down:** the hand loop above trained a plain `Linear` for three
epochs. For this slightly larger model, at which epoch do you expect the valid loss to be
lowest: before 5, around 15, or not before 30? And which of the three shapes do you expect?

In [ ]:
import numpy as np

torch.manual_seed(42)
_, full, full_secs = train(Serial([NeuralNet(hidden_size=HIDDEN, out_size=N_CLASSES)]))

lowest = int(np.argmin(full["valid_loss"]))
lr_drop = next((e for e, lr in enumerate(full["lr"]) if lr < full["lr"][0]), None)
print(f"{len(full['valid_loss'])} epochs in {full_secs:.1f}s")
print(f"valid loss lowest at epoch {lowest}: {full['valid_loss'][lowest]:.4f}")
print(f"valid loss at the last epoch:     {full['valid_loss'][-1]:.4f}")
print(
    f"train loss at epoch {lowest} -> last: {full['train_loss'][lowest]:.4f} -> {full['train_loss'][-1]:.4f}"
)
print("valid loss, epochs 0-9:", " ".join(f"{v:.3f}" for v in full["valid_loss"][:10]))
print(f"learning rate {full['lr'][0]:g} until epoch {lr_drop}, then {full['lr'][-1]:g}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(full["train_loss"], color="tab:blue", marker=".", label="train loss")
ax.plot(full["valid_loss"], color="tab:orange", marker=".", label="valid loss")
ax.set_xlabel("epoch (0 = the first pass over the train rows)")
ax.set_ylabel("cross-entropy loss")
ax.set_title("Train loss keeps falling; valid loss bottoms out early, then rises")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The valid loss is lowest at **epoch 3** (0.167) and then climbs to 0.22 by epoch
9, while the train loss keeps falling, from 0.072 at epoch 3 to 0.001 at the last epoch.
That is the third shape: **overfitting**. After epoch 3 the head gets better at the 4800
training rows and worse at the 1200 rows it does not train on. The 26 epochs after it
bought nothing on the valid split.

From about epoch 11 both curves go flat. The train loss is near zero, so the train rows
have little left to teach, and the scheduler `ReduceLROnPlateau` in `train` lowers the
learning rate once the valid loss has not improved for 10 epochs (0.001 until epoch 14,
1e-05 at the end), so the weights hardly move any more. Flat here comes after the turn;
it does not undo it.

Had the valid loss still been falling at epoch 29, the reading would have been the
opposite: the budget was too short and the run proves nothing about overfitting. Look at
the first epochs too: the valid loss goes 0.190 at epoch 1, **up** to 0.200 at epoch 2,
then down to its lowest at epoch 3. One epoch up is not yet a turn. §3.5 needs that.

## 3.5 Early stopping: let the valid loss decide

The curve already told you when to stop. Early stopping is that reading done by the loop
itself, every epoch, so no epoch count has to be picked by hand:

1. after each epoch, compute the valid loss;
2. if it is the lowest so far, copy the model's weights and set a counter to 0;
3. if it is not, add 1 to the counter;
4. when the counter reaches the **patience**, stop, and put back the copied weights.

`epochs` then stops being *the* number of epochs and becomes the most a run may take. You
get back the weights of the best epoch, not the last one. Note what it watches: the valid
*loss*, the number the optimizer works on, not the accuracy.

Why not stop at the first epoch that is worse? Because the valid loss wobbles: 1200 rows
give a noisy number, and a single bad epoch can be followed by a new low. The patience is
how many bad epochs in a row you accept before you believe the curve has turned. Too short
and a wobble stops you; too long and you pay for epochs you throw away.

The function `where_it_stops` applies those four steps to a curve you already have, so you
can see where a run would stop without training it again.

In [ ]:
def where_it_stops(valid_loss: list[float], patience: int) -> tuple[int, int]:
    """(best epoch, last epoch that runs) under early stopping with this patience."""
    best, counter = 0, 0
    for epoch, loss in enumerate(valid_loss):
        if loss < valid_loss[best]:
            best, counter = epoch, 0
        elif epoch > 0:
            counter += 1
            if counter == patience:
                return best, epoch
    return best, len(valid_loss) - 1  # never ran out of patience: the budget ended it


PATIENCE = (
    5  # longer than one wobble of the valid loss, short enough to waste few epochs
)

best, stop = where_it_stops(full["valid_loss"], PATIENCE)
print(f"patience {PATIENCE}: best epoch {best}, stops after epoch {stop}")

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(full["train_loss"], color="tab:blue", marker=".", label="train loss")
ax.plot(full["valid_loss"], color="tab:orange", marker=".", label="valid loss")
ax.axvspan(
    best, stop, color="tab:orange", alpha=0.12, label=f"{PATIENCE} epochs of patience"
)
ax.axvspan(
    stop, len(full["valid_loss"]) - 1, color="grey", alpha=0.12, label="never run"
)
ax.scatter(
    [best],
    [full["valid_loss"][best]],
    color="tab:red",
    zorder=3,
    s=60,
    label=f"best epoch {best}: these weights are kept",
)
ax.axvline(stop, color="black", lw=1, ls="--", label=f"stop after epoch {stop}")
ax.set_xlabel("epoch (0 = the first pass over the train rows)")
ax.set_ylabel("cross-entropy loss")
ax.set_title(
    f"Early stopping with patience {PATIENCE}: keep the red dot, stop at the dashed line"
)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

The red dot is epoch 3, the lowest valid loss; those are the weights early stopping
keeps. Five epochs without a new low follow (the orange band, epochs 4 to 8), so the run
stops after epoch 8, and the 21 grey epochs are never trained. The curve-reading of §3.4
and early stopping agree on the epoch: early stopping only saves you the reading and the
grey epochs.

With a patience of 1 the wobble at epoch 2 would have ended the run there, keeping
epoch 1's weights (valid loss 0.190 instead of 0.167). That is why the patience is longer
than one epoch. The exercise E5 asks you to try others.

## 3.6 The baseline

Now the number. `NeuralNet(hidden_size, out_size)` is two `Linear` layers with a `GELU`
between them — notebook 02's construction, once. `Serial` chains components; with a single
component it does nothing at all, and it is written this way because notebook 04 will add
things to that list.

It trains with early stopping, `PATIENCE` = 5, at most `EPOCHS` = 30. **Predict first,
from the picture above:** at which epoch will it keep its weights, and after which epoch
will it stop? Then run it. The valid accuracy is the score at the best epoch; the test
accuracy is computed once, on the weights early stopping kept, and is the number you
write down.

This is the model to beat.

In [ ]:
torch.manual_seed(42)
baseline = Serial([NeuralNet(hidden_size=HIDDEN, out_size=N_CLASSES)])

result, curves, secs = train(baseline, patience=PATIENCE)
params = sum(p.numel() for p in baseline.parameters() if p.requires_grad)
test_accuracy = score(baseline, testloader)  # the one look at test

print(
    f"best epoch {result.epoch}, stopped after epoch {result.stopped_epoch} (of at most {EPOCHS})"
)
print(
    f"valid loss {result.test_loss:.4f}   valid accuracy {result.metric_dict['Accuracy']:.4f}"
)
print(
    f"\nBASELINE  test accuracy {test_accuracy:.4f}   {params:,} parameters   {secs:.1f}s"
)

Write that number down, together with how it was trained: best epoch 3, stopped after
epoch 8, as the picture of §3.5 predicted from the full curve. The valid accuracy at the
best epoch is 0.943; the test accuracy, looked at once, is **0.949**.

**It cost seconds.** Not because the problem is easy — because you are training over cached
vectors instead of images. The same accuracy from pixels means a CNN, a GPU, and a coffee
break. That asymmetry is the entire reason this course is arranged the way it is. Early
stopping made it cheaper still: 9 epochs instead of 30.

**It is already close to the ceiling.** The measured best for a small head on this cache is
around 0.95. Whatever notebook 04's search finds, it is fighting for the last few points,
and you should expect most of its trials to land *below* this line rather than above it.

**It is a small model.** One hidden layer you did not choose, no fusion, no gating, no
tuning. Every architecture you are about to try has to earn the difference — and the
honest default expectation is that most of them will not.

That is not cynicism, it is the reason the search is worth running: you find out, in
seconds, instead of arguing about it. Notebook 04 does exactly that, on a dataset with more
headroom than this one.

## What you now know

- A collate function turns a list of items into `(X, y)`. Stacking is enough for `(D)`
  items; padding is for the chunked case.
- The loss defines the task: `CrossEntropy` when classes compete, `BCEWithLogits` when they
  do not. Feed both of them logits.
- The optimizer decides the step. The learning rate has to fit the loss surface: too small
  crawls, too large diverges. Momentum and Adam help along flat directions.
- The loop is `zero_grad → forward → loss → backward → step`, and `eval()`/`no_grad()`
  around evaluation.
- Three splits: train to learn from, valid to decide on, test to look at once. A split you
  decide on gives a score that is a little too good, so it cannot be the one you report.
- A loss curve says whether a model is still learning (both falling), finished (both
  flat) or overfitting (train falling, valid rising). This head overfits after epoch 3.
- Early stopping reads that curve for you: it keeps the weights of the lowest valid loss and
  stops after `patience` epochs without a new low. `epochs` becomes a maximum; the patience
  has to outlast a wobble of the valid loss.
- You have a baseline on EuroSAT, established *before* any searching, with its best and
  stopped epoch next to its score. That is what makes the next notebook readable.

## Exercises

**E1 — the wrong loss on purpose.** Swap `CrossEntropyLoss` for `BCEWithLogitsLoss` on this
single-label task and retrain. Accuracy barely moves. Explain why the metric fails to
notice, and what would have to be true about EuroSAT for the swap to be *correct* rather
than merely survivable.

**E2 — where does the gap come from?** Shrink the training rows to 100
(`trainset.select(range(100))`), build a loader for them the way §3.1 does, and retrain
with `train(..., patience=PATIENCE, trainloader=small_loader)`. Which moves more: the
training loss, or the gap between train and valid loss? And does early stopping stop
earlier or later than at epoch 8? Predict before running, then say which of the numbers
tells you that you are overfitting.

**E3 — skip a line.** Delete `optimizer.zero_grad()` from the hand-written loop in §3.3 and
run it. Describe what the loss curve does and explain it in terms of accumulated gradients.
(This is the single most common torch bug; meeting it deliberately once is cheaper than
meeting it accidentally at 2am.)

**E4 — is your baseline stable?** Rerun §3.6 with three different seeds and print the
spread of the test accuracy, and the best epoch of each run. You now know how big a "win"
has to be in notebook 04 before it means anything.

**E5 — change the patience.** Before running anything, read the curve of §3.4 and write
down the best epoch and the stop epoch for a patience of 1, 2 and 10. Check your reading
with `where_it_stops(full["valid_loss"], p)`, then train the baseline with
`train(..., patience=p)` for each and compare `result.epoch` and `result.stopped_epoch`
with your prediction. Which patience kept the worst weights, and which spent the most
epochs? What would you need to see in a curve to choose a patience longer than 5?